In [7]:
"""
GMD-capable workflow finder (GitHub Actions) — CSV in / CSV out (same folder)

- Input : C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\GMD_Check\GMD_URLs.csv
- Output: C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\GMD_Check\gmd_workflow_scan_results.csv

Reads a CSV file of GitHub URLs, extracts repos, scans .github/workflows/*.yml|*.yaml,
and flags repos/workflows as "GMD-capable" using conservative heuristics:

GMD-capable if ANY workflow contains at least one of:
  A) Gradle invocation of a Managed Device task (pixel2Api32DebugAndroidTest, *managedDevice*AndroidTest, *allDevices*AndroidTest, *Api<digits>*AndroidTest)
  B) Managed Devices properties (-Pandroid.testoptions.manageddevices.* or -Pandroid.experimental.testOptions.managedDevices.*)
  C) Managed Device report paths (build/reports/androidTests/managedDevice / managedDevice/allDevices)

Token loading:
  TOKENS_ENV_PATH points to:
    GITHUB_TOKEN_1=...
    GITHUB_TOKEN_2=...
    ...

The scanner rotates tokens (round-robin), retries on rate limits.
"""

import json
import re
import time
from dataclasses import dataclass
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

# =========================
# CONFIG (as requested)
# =========================
BASE_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\GMD_Check")
INPUT_CSV = BASE_DIR / "GMD_URLs.csv"
OUTPUT_CSV = BASE_DIR / "gmd_workflow_scan_results.csv"

TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
MAX_TOKENS_TO_USE = 3

URL_COL_HINTS = ["url", "urls", "link", "links", "github", "repo", "repository"]

SLEEP_BETWEEN_REPOS_SEC = 0.20
MAX_RETRIES = 6
TIMEOUT = 30

WF_EXTS = (".yml", ".yaml")

# =========================
# Heuristics (GMD) — tighter to avoid false positives
# =========================
RE_GRADLE_CMD = re.compile(r"(\./gradlew\b|gradlew\.bat\b|\bgradle\s+)", re.IGNORECASE)

RE_MANAGEDDEVICES_PROP = re.compile(
    r"-Pandroid\.(?:testoptions\.manageddevices|experimental\.testOptions\.managedDevices)\.[\w\.\-]+",
    re.IGNORECASE
)

RE_GMD_REPORT_PATH = re.compile(
    r"(build/reports/androidtests/manageddevice|manageddevice/alldevices|androidtests/manageddevice)",
    re.IGNORECASE
)

# Strong GMD task indicators:
# - managedDevice*AndroidTest
# - allDevices*AndroidTest
# - <name>Api<digits>*AndroidTest (e.g., pixel2Api32DebugAndroidTest)
RE_GMD_TASK_STRONG = re.compile(
    r"""
    (?:
        \bmanageddevice[\w:.-]*androidtest\b
        |
        \balldevices[\w:.-]*androidtest\b
        |
        \b[a-z0-9_-]+api\d+[\w:.-]*androidtest\b
    )
    """,
    re.IGNORECASE | re.VERBOSE
)

RE_EXCLUDE_CONNECTED = re.compile(r"\bconnected[\w:.-]*androidtest\b", re.IGNORECASE)
RE_EXCLUDE_SPOON_MARATHON = re.compile(r"\b(spoon|marathon)\b", re.IGNORECASE)


# =========================
# Token loading (same style as your Stage3 code)
# =========================
def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3):
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens


# =========================
# GitHub API client (rotating tokens)
# =========================
@dataclass
class TokenState:
    token: str
    remaining: int | None = None
    reset_epoch: int | None = None


class GitHubClient:
    def __init__(self, tokens):
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "gmd-capable-scanner/1.3",
        })
        self.tokens = [TokenState(t) for t in tokens]
        self._rr = 0

    def _pick_token(self):
        n = len(self.tokens)
        for _ in range(n):
            st = self.tokens[self._rr % n]
            self._rr += 1
            if st.remaining is None or st.remaining > 0:
                return st
        return self.tokens[0]

    def _sleep_until_reset(self):
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def request(self, method, url, params=None):
        last_status = None
        for attempt in range(1, MAX_RETRIES + 1):
            st = self._pick_token()
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                r = self.session.request(method, url, params=params, timeout=TIMEOUT)
            except requests.RequestException:
                time.sleep(min(60, 2 ** attempt))
                continue

            last_status = r.status_code

            rem = r.headers.get("X-RateLimit-Remaining")
            rst = r.headers.get("X-RateLimit-Reset")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if r.status_code in (200, 404):
                return r

            txt_l = (r.text or "").lower()
            if r.status_code in (403, 429) and ("rate limit" in txt_l or "secondary rate limit" in txt_l):
                known = [ts.remaining for ts in self.tokens if ts.remaining is not None]
                if known and all(x == 0 for x in known):
                    self._sleep_until_reset()
                else:
                    time.sleep(min(60, 2 ** attempt))
                continue

            if r.status_code in (500, 502, 503, 504):
                time.sleep(min(60, 2 ** attempt))
                continue

            return r

        raise RuntimeError(f"Failed {method} after retries: {url} (last_status={last_status})")

    def get_json(self, url, params=None):
        r = self.request("GET", url, params=params)
        if r.status_code == 404:
            return None
        try:
            return r.json()
        except Exception:
            return None

    def get_text(self, url, params=None):
        r = self.request("GET", url, params=params)
        if r.status_code == 404:
            return ""
        return r.text or ""


# =========================
# Repo parsing + workflow listing
# =========================
def parse_repo_from_url(u: str):
    if not isinstance(u, str) or not u.strip():
        return None
    u = u.strip()

    if re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", u):
        owner, repo = u.split("/", 1)
        return owner, repo

    try:
        p = urlparse(u)
        if p.netloc.lower() != "github.com":
            return None
        parts = [x for x in p.path.split("/") if x]
        if len(parts) >= 2:
            return parts[0], parts[1]
    except Exception:
        return None
    return None


def list_workflows(gh: GitHubClient, owner: str, repo: str):
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/.github/workflows"
    data = gh.get_json(url)
    if not isinstance(data, list):
        return []
    out = []
    for item in data:
        name = (item.get("name") or "")
        if name.lower().endswith(WF_EXTS):
            out.append({
                "path": item.get("path", ""),
                "name": name,
                "download_url": item.get("download_url", ""),
            })
    return out


# =========================
# Detection
# =========================
def detect_gmd_in_workflow_text(yaml_text: str):
    if not yaml_text:
        return {"gmd_capable": False, "reasons": [], "examples": []}

    reasons = []
    examples = []

    m = RE_MANAGEDDEVICES_PROP.search(yaml_text)
    if m:
        reasons.append("manageddevices_properties")
        examples.append(yaml_text[max(0, m.start()-40): m.end()+40].replace("\n", " ")[:220])

    m = RE_GMD_REPORT_PATH.search(yaml_text)
    if m:
        reasons.append("manageddevice_report_paths")
        examples.append(yaml_text[max(0, m.start()-40): m.end()+40].replace("\n", " ")[:220])

    for line in yaml_text.splitlines():
        if not RE_GRADLE_CMD.search(line):
            continue
        if RE_EXCLUDE_SPOON_MARATHON.search(line):
            continue
        if RE_EXCLUDE_CONNECTED.search(line):
            continue
        if RE_GMD_TASK_STRONG.search(line):
            reasons.append("gradle_gmd_task_invocation")
            examples.append(line.strip()[:220])
            break

    return {"gmd_capable": bool(reasons), "reasons": sorted(set(reasons)), "examples": examples[:3]}


# =========================
# IO helpers
# =========================
def find_url_column(df: pd.DataFrame):
    cols_lower = {str(c).lower(): c for c in df.columns}
    for h in URL_COL_HINTS:
        if h in cols_lower:
            return cols_lower[h]
    # fallback heuristic
    for c in df.columns:
        s = df[c].astype(str)
        if s.str.contains("github.com", case=False, na=False).mean() > 0.2:
            return c
    return df.columns[0]


# =========================
# Main
# =========================
def main():
    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    df_in = pd.read_csv(INPUT_CSV, encoding="utf-8", engine="python")
    url_col = find_url_column(df_in)

    repos = []
    for u in df_in[url_col].astype(str).tolist():
        parsed = parse_repo_from_url(u)
        if parsed:
            repos.append(parsed)

    if not repos:
        raise RuntimeError(f"No GitHub repos found in column '{url_col}'. Check {INPUT_CSV}.")

    results = []
    seen = set()

    for owner, repo in repos:
        key = (owner.lower(), repo.lower())
        if key in seen:
            continue
        seen.add(key)

        wf_files = list_workflows(gh, owner, repo)

        repo_capable = False
        repo_reasons = set()
        matched = []

        for wf in wf_files:
            txt = gh.get_text(wf["download_url"]) if wf.get("download_url") else ""
            det = detect_gmd_in_workflow_text(txt)
            if det["gmd_capable"]:
                repo_capable = True
                repo_reasons.update(det["reasons"])
                matched.append({
                    "workflow_path": wf["path"],
                    "workflow_name": wf["name"],
                    "reasons": det["reasons"],
                    "examples": det["examples"],
                })

        results.append({
            "owner": owner,
            "repo": repo,
            "repo_url": f"https://github.com/{owner}/{repo}",
            "gmd_capable": repo_capable,
            "repo_reasons": ", ".join(sorted(repo_reasons)) if repo_reasons else "",
            "matched_workflow_count": len(matched),
            "matched_workflows_json": json.dumps(matched, ensure_ascii=False),
        })

        time.sleep(SLEEP_BETWEEN_REPOS_SEC)

    df_out = pd.DataFrame(results).sort_values(
        ["gmd_capable", "matched_workflow_count", "owner", "repo"],
        ascending=[False, False, True, True]
    ).reset_index(drop=True)

    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"[done] wrote: {OUTPUT_CSV}")
    print(f"[done] scanned repos: {len(df_out)}")
    print(f"[done] gmd_capable repos: {int(df_out['gmd_capable'].sum())}")


if __name__ == "__main__":
    main()


[done] wrote: C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\GMD_Check\gmd_workflow_scan_results.csv
[done] scanned repos: 17
[done] gmd_capable repos: 7


In [9]:
"""
GMD workflow verifier — Run inventory + jobs + steps + enhanced metrics (UNBOUNDED PAST, cutoff at Feb 10, 2026)

What it does
------------
Input:
  - gmd_workflow_targets.csv (or any CSV) containing GitHub repo/workflow references.
    Expected columns (best-effort detection):
      - repo_url OR repo OR full_name OR owner/repo
      - workflow_path OR workflow_file OR workflow_id (any is fine)

Output (written to OUT_DIR, same folder as input):
  - gmd_run_inventory.csv      : run-level inventory for all runs <= cutoff
  - gmd_run_jobs_raw.csv       : raw jobs payload flattened (one row per job)
  - gmd_run_steps_raw.csv      : raw steps flattened (one row per step)
  - gmd_run_metrics_enhanced.csv : run-level metrics computed from jobs/steps
  - gmd_run_inventory_log.csv  : per-workflow debug stats (seen/kept/skipped)

Key behaviors
-------------
- NO start date bound (collect as far back as GitHub returns via pagination)
- Inclusive cutoff end date: 2026-02-10T23:59:59Z
- Includes workflow_dispatch runs (no event filtering)
- IMPORTANT: Uses workflow file NAME (e.g., instru_test_GMD.yml), not full path

Auth
----
Reads GitHub tokens from:
  TOKENS_ENV_PATH = r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env"
using keys like:
  GITHUB_TOKEN_1=...
  GITHUB_TOKEN_2=...

Notes
-----
- GitHub API pagination is by page/per_page. This script paginates until empty.
- For large repos, this can be heavy. It’s “unbounded past” by design.
"""

from __future__ import annotations

import csv
import json
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple
from urllib.parse import urlparse

import pandas as pd
import requests

# =========================
# CONFIG
# =========================
INPUT_TARGETS_CSV = r"C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\GMD_Check\gmd_workflow_targets.csv"
OUT_DIR = str(Path(INPUT_TARGETS_CSV).parent)

TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
MAX_TOKENS_TO_USE = 3

CUTOFF_MAX_ISO = "2026-02-10T23:59:59Z"  # inclusive
PER_PAGE = 100
MAX_PAGES_PER_WORKFLOW = 10_000  # practical "no limit"; stops when API returns empty

SLEEP_BETWEEN_REQUESTS_SEC = 0.15
MAX_RETRIES = 8
TIMEOUT = 45

# Column detection hints (input CSV)
REPO_COL_HINTS = ["repo_url", "repository", "repo", "full_name", "owner_repo", "owner/repo", "url"]
WF_PATH_HINTS = ["workflow_path", "workflow_file", "workflow", "workflow_yml", "workflow_yaml", "wf_path"]
WF_ID_HINTS = ["workflow_id", "wf_id"]

# =========================
# Helpers
# =========================
def iso_to_dt_z(s: str) -> Optional[datetime]:
    if not s:
        return None
    try:
        return datetime.fromisoformat(s.replace("Z", "+00:00")).astimezone(timezone.utc)
    except Exception:
        return None

CUTOFF_MAX_DT = iso_to_dt_z(CUTOFF_MAX_ISO)
if CUTOFF_MAX_DT is None:
    raise ValueError("Invalid CUTOFF_MAX_ISO")

def safe_mkdir(p: str) -> None:
    Path(p).mkdir(parents=True, exist_ok=True)

def write_csv_rows(path: str, fieldnames: List[str], rows: Iterable[Dict]) -> None:
    safe_mkdir(str(Path(path).parent))
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)

def append_csv_rows(path: str, fieldnames: List[str], rows: Iterable[Dict]) -> None:
    exists = Path(path).exists()
    safe_mkdir(str(Path(path).parent))
    with open(path, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        if not exists:
            w.writeheader()
        for r in rows:
            w.writerow(r)

def pick_first_existing_col(df: pd.DataFrame, hints: List[str]) -> Optional[str]:
    cols_lower = {str(c).lower(): c for c in df.columns}
    for h in hints:
        if h.lower() in cols_lower:
            return cols_lower[h.lower()]
    return None

def parse_owner_repo(x: str) -> Optional[Tuple[str, str]]:
    if not isinstance(x, str) or not x.strip():
        return None
    s = x.strip()

    # already "owner/repo"
    if re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", s):
        owner, repo = s.split("/", 1)
        return owner, repo

    # URL
    try:
        p = urlparse(s)
        if p.netloc.lower() == "github.com":
            parts = [t for t in p.path.split("/") if t]
            if len(parts) >= 2:
                return parts[0], parts[1]
    except Exception:
        pass
    return None

def workflow_ref_from_row(row: Dict) -> Optional[str]:
    """
    Prefer workflow_id if present; else workflow file name; else parse from workflow_path.
    IMPORTANT: if workflow_path includes ".github/workflows/...", we return just filename.
    """
    for k in WF_ID_HINTS:
        if k in row and str(row[k]).strip():
            return str(row[k]).strip()  # numeric string ok for API

    for k in WF_PATH_HINTS:
        if k in row and str(row[k]).strip():
            v = str(row[k]).strip()
            # if full path, reduce to file name
            return Path(v).name

    return None

# =========================
# Token loading
# =========================
def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

# =========================
# GitHub API client (rotating tokens)
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "gmd-unbounded-inventory/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]
        self._rr = 0

    def _pick_token(self) -> TokenState:
        n = len(self.tokens)
        for _ in range(n):
            st = self.tokens[self._rr % n]
            self._rr += 1
            if st.remaining is None or st.remaining > 0:
                return st
        return self.tokens[0]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def request(self, method: str, url: str, params: Optional[Dict] = None) -> requests.Response:
        last_status = None
        for attempt in range(1, MAX_RETRIES + 1):
            st = self._pick_token()
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                r = self.session.request(method, url, params=params, timeout=TIMEOUT)
            except requests.RequestException:
                time.sleep(min(60, 2 ** attempt))
                continue

            last_status = r.status_code

            rem = r.headers.get("X-RateLimit-Remaining")
            rst = r.headers.get("X-RateLimit-Reset")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if r.status_code in (200, 404):
                return r

            txt_l = (r.text or "").lower()
            if r.status_code in (403, 429) and ("rate limit" in txt_l or "secondary rate limit" in txt_l):
                # if all tokens appear exhausted, sleep
                if all(ts.remaining == 0 for ts in self.tokens if ts.remaining is not None):
                    self._sleep_until_reset()
                else:
                    time.sleep(min(60, 2 ** attempt))
                continue

            if r.status_code in (500, 502, 503, 504):
                time.sleep(min(60, 2 ** attempt))
                continue

            # other 4xx -> return
            return r

        raise RuntimeError(f"Failed {method} after retries: {url} (last_status={last_status})")

    def get_json(self, url: str, params: Optional[Dict] = None) -> Optional[Dict]:
        r = self.request("GET", url, params=params)
        if r.status_code == 404:
            return None
        try:
            return r.json()
        except Exception:
            return None

# =========================
# Core extractors
# =========================
def list_all_workflow_runs_unbounded_past(
    gh: GitHubClient,
    owner: str,
    repo: str,
    workflow_ref: str,          # workflow file name OR workflow id
    cutoff_max_dt: datetime,
) -> Tuple[List[Dict], Dict]:
    """
    Fetch all runs for a workflow, going as far back as GitHub provides.
    Keep only runs with created_at <= cutoff_max_dt.
    """
    url = f"https://api.github.com/repos/{owner}/{repo}/actions/workflows/{workflow_ref}/runs"

    kept: List[Dict] = []
    page = 1
    total_seen = 0
    skipped_newer = 0
    status_404 = False

    while page <= MAX_PAGES_PER_WORKFLOW:
        data = gh.get_json(url, params={"per_page": PER_PAGE, "page": page})
        if data is None:
            # Could be 404 (workflow not found) or other failure
            # We can't directly see status code here, but None typically means 404 or json failure.
            status_404 = True
            break

        runs = data.get("workflow_runs") or []
        if not runs:
            break

        for r in runs:
            total_seen += 1
            created = iso_to_dt_z(r.get("created_at", ""))
            if created is None:
                continue
            if created <= cutoff_max_dt:
                kept.append(r)
            else:
                skipped_newer += 1

        page += 1
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    stats = {
        "workflow_ref": workflow_ref,
        "total_seen": total_seen,
        "kept": len(kept),
        "skipped_newer": skipped_newer,
        "ended_at_page": page,
        "possible_404_or_bad_ref": status_404 and total_seen == 0,
        "runs_endpoint": url,
    }
    return kept, stats

def list_run_jobs(
    gh: GitHubClient,
    owner: str,
    repo: str,
    run_id: int,
) -> List[Dict]:
    """
    Paginate /actions/runs/{run_id}/jobs (jobs include steps).
    """
    url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs/{run_id}/jobs"
    out: List[Dict] = []
    page = 1
    while page <= 10_000:
        data = gh.get_json(url, params={"per_page": PER_PAGE, "page": page})
        if not data:
            break
        jobs = data.get("jobs") or []
        if not jobs:
            break
        out.extend(jobs)
        if len(jobs) < PER_PAGE:
            break
        page += 1
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)
    return out

def compute_enhanced_metrics_from_jobs(run: Dict, jobs: List[Dict]) -> Dict:
    """
    Basic metrics aligned with your Stage3 definitions:
      - queue_seconds = run_started_at - created_at
      - run_duration_seconds = max(job_completed) - min(job_started)
      - instru_job_count / instru_step_count based on simple instrumentation heuristics
        (kept minimal; you're already doing richer detection elsewhere)
    """
    created = iso_to_dt_z(run.get("created_at", ""))
    started = iso_to_dt_z(run.get("run_started_at", ""))

    queue_seconds = None
    if created and started:
        qs = int((started - created).total_seconds())
        queue_seconds = qs if qs >= 0 else None

    starts: List[datetime] = []
    ends: List[datetime] = []
    runner_labels = set()

    # Minimal instrumentation heuristic (you can replace with your exact regex set)
    test_re = re.compile(r"(androidtest|connectedcheck|connectedandroidtest|manageddevice|gmd|instrument)", re.I)
    env_re = re.compile(r"(emulator|avd|adb|kvm|sdkmanager|avdmanager)", re.I)

    instru_job_names = []
    instru_step_names = []

    for j in jobs:
        js = iso_to_dt_z(j.get("started_at", "") or "")
        je = iso_to_dt_z(j.get("completed_at", "") or "")
        if js:
            starts.append(js)
        if je:
            ends.append(je)
        for lab in (j.get("labels") or []):
            if isinstance(lab, str) and lab.strip():
                runner_labels.add(lab.strip())

        job_name = (j.get("name") or "").strip()
        steps = j.get("steps") if isinstance(j.get("steps"), list) else []

        job_instruish = bool(test_re.search(job_name) or env_re.search(job_name))
        if not job_instruish:
            job_instruish = any(test_re.search((st.get("name") or "")) or env_re.search((st.get("name") or "")) for st in steps)

        if job_instruish and job_name:
            instru_job_names.append(job_name)

        for st in steps:
            sn = (st.get("name") or "").strip()
            if not sn:
                continue
            if test_re.search(sn):
                instru_step_names.append(sn)

    run_duration_seconds = None
    if starts and ends:
        rd = int((max(ends) - min(starts)).total_seconds())
        run_duration_seconds = rd if rd >= 0 else None

    return {
        "queue_seconds": queue_seconds,
        "run_duration_seconds": run_duration_seconds,
        "runner_labels_union": ",".join(sorted(runner_labels)),
        "instru_job_count": len({x for x in instru_job_names if x}),
        "instru_step_count": len({x for x in instru_step_names if x}),
        "instru_job_names": ",".join(sorted({x for x in instru_job_names if x}))[:800],
        "instru_step_names": ",".join(sorted({x for x in instru_step_names if x}))[:800],
    }

# =========================
# Main
# =========================
def main() -> None:
    print("[debug] input:", INPUT_TARGETS_CSV)
    print("[debug] outdir:", OUT_DIR)
    print("[debug] cutoff_max:", CUTOFF_MAX_ISO)

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    df = pd.read_csv(INPUT_TARGETS_CSV)

    repo_col = pick_first_existing_col(df, REPO_COL_HINTS)
    if repo_col is None:
        repo_col = df.columns[0]

    # We'll keep any workflow-related columns present
    # but we need at least one of: workflow_id OR workflow_path/file
    results_log = []

    run_inventory_rows: List[Dict] = []
    jobs_rows: List[Dict] = []
    steps_rows: List[Dict] = []
    metrics_rows: List[Dict] = []

    seen_workflows = set()

    for i, row in df.iterrows():
        repo_val = str(row.get(repo_col, "")).strip()
        parsed = parse_owner_repo(repo_val)
        if not parsed:
            continue
        owner, repo = parsed

        row_dict = {str(k): ("" if pd.isna(v) else v) for k, v in row.to_dict().items()}
        wf_ref = workflow_ref_from_row(row_dict)
        if not wf_ref:
            continue

        wf_key = (owner.lower(), repo.lower(), str(wf_ref).lower())
        if wf_key in seen_workflows:
            continue
        seen_workflows.add(wf_key)

        runs, stats = list_all_workflow_runs_unbounded_past(gh, owner, repo, wf_ref, CUTOFF_MAX_DT)
        stats.update({"owner": owner, "repo": repo, "repo_url": f"https://github.com/{owner}/{repo}"})
        results_log.append(stats)

        print(f"[debug] {owner}/{repo} wf={wf_ref} seen={stats['total_seen']} kept={stats['kept']} skipped_newer={stats['skipped_newer']}")

        for r in runs:
            run_id = r.get("id")
            if not run_id:
                continue

            run_inventory_rows.append({
                "owner": owner,
                "repo": repo,
                "repo_url": f"https://github.com/{owner}/{repo}",
                "workflow_ref": wf_ref,
                "run_id": run_id,
                "run_number": r.get("run_number", ""),
                "run_attempt": r.get("run_attempt", ""),
                "event": r.get("event", ""),
                "status": r.get("status", ""),
                "conclusion": r.get("conclusion", ""),
                "created_at": r.get("created_at", ""),
                "run_started_at": r.get("run_started_at", ""),
                "updated_at": r.get("updated_at", ""),
                "head_branch": r.get("head_branch", ""),
                "head_sha": r.get("head_sha", ""),
                "html_url": r.get("html_url", ""),
            })

            # jobs + steps
            jobs = list_run_jobs(gh, owner, repo, int(run_id))

            # flatten jobs
            for j in jobs:
                jobs_rows.append({
                    "owner": owner,
                    "repo": repo,
                    "run_id": run_id,
                    "job_id": j.get("id", ""),
                    "job_name": j.get("name", ""),
                    "job_status": j.get("status", ""),
                    "job_conclusion": j.get("conclusion", ""),
                    "job_started_at": j.get("started_at", ""),
                    "job_completed_at": j.get("completed_at", ""),
                    "runner_name": j.get("runner_name", ""),
                    "labels": ",".join(j.get("labels") or []),
                    "raw_job_json": json.dumps(j, ensure_ascii=False)[:20000],
                })

                steps = j.get("steps") if isinstance(j.get("steps"), list) else []
                for st in steps:
                    steps_rows.append({
                        "owner": owner,
                        "repo": repo,
                        "run_id": run_id,
                        "job_id": j.get("id", ""),
                        "job_name": j.get("name", ""),
                        "step_number": st.get("number", ""),
                        "step_name": st.get("name", ""),
                        "step_status": st.get("status", ""),
                        "step_conclusion": st.get("conclusion", ""),
                        "step_started_at": st.get("started_at", ""),
                        "step_completed_at": st.get("completed_at", ""),
                        "raw_step_json": json.dumps(st, ensure_ascii=False)[:20000],
                    })

            # enhanced metrics
            m = compute_enhanced_metrics_from_jobs(r, jobs)
            metrics_rows.append({
                "owner": owner,
                "repo": repo,
                "run_id": run_id,
                "workflow_ref": wf_ref,
                "event": r.get("event", ""),
                "conclusion": r.get("conclusion", ""),
                "created_at": r.get("created_at", ""),
                "run_started_at": r.get("run_started_at", ""),
                "html_url": r.get("html_url", ""),
                **m,
            })

            time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

    # Write outputs
    out_inventory = str(Path(OUT_DIR) / "gmd_run_inventory.csv")
    out_jobs = str(Path(OUT_DIR) / "gmd_run_jobs_raw.csv")
    out_steps = str(Path(OUT_DIR) / "gmd_run_steps_raw.csv")
    out_metrics = str(Path(OUT_DIR) / "gmd_run_metrics_enhanced.csv")
    out_log = str(Path(OUT_DIR) / "gmd_run_inventory_log.csv")

    pd.DataFrame(results_log).to_csv(out_log, index=False, encoding="utf-8")
    pd.DataFrame(run_inventory_rows).to_csv(out_inventory, index=False, encoding="utf-8")
    pd.DataFrame(jobs_rows).to_csv(out_jobs, index=False, encoding="utf-8")
    pd.DataFrame(steps_rows).to_csv(out_steps, index=False, encoding="utf-8")
    pd.DataFrame(metrics_rows).to_csv(out_metrics, index=False, encoding="utf-8")

    print("[done] wrote:", out_log)
    print("[done] wrote:", out_inventory)
    print("[done] wrote:", out_jobs)
    print("[done] wrote:", out_steps)
    print("[done] wrote:", out_metrics)
    print("[done] workflows scanned:", len(results_log))
    print("[done] runs kept:", len(run_inventory_rows))

if __name__ == "__main__":
    main()


[debug] input: C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\GMD_Check\gmd_workflow_targets.csv
[debug] outdir: C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\GMD_Check
[debug] cutoff_max: 2026-02-10T23:59:59Z
[debug] FliegendeWurst/TriliumDroid wf=test.yaml seen=228 kept=226 skipped_newer=2
[debug] NUmeroAndDev/MaterialGallery-android wf=baseline-profiles.yml seen=0 kept=0 skipped_newer=0
[debug] behnamparsa/toDoList wf=instru_test_GMD.yml seen=13 kept=13 skipped_newer=0
[debug] flauschtrud/broccoli wf=build.yml seen=72 kept=72 skipped_newer=0
[debug] irgaly/kfswatch wf=build-test.yml seen=207 kept=207 skipped_newer=0
[debug] ryanw-mobile/OctoMeter wf=renovate_check.yml seen=868 kept=851 skipped_newer=17
[debug] sopt-makers/sopt-android wf=baseline-profile.yml seen=0 kept=0 skipped_newer=0
[done] wrote: C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\GMD_Check\gmd_run_inventory_log.csv
[done] wrote: C:\Android Mobile App\ICST2026_Ext\0_Data_Feb_10\GMD_Check\gmd_run_inventory.csv